# 13 — IL-8 / CXCL8 across ALL conditions (focus decision point)

Figure 1: **CXCL8 (IL-8) only**, log2FC across every available contrast in both studies.
Figure 2: **HSPA5 on its own** (separate graphic — it's here because it correlates with
CXCL8 at ρ=0.92, not because it's an IL-8 comparison).

Reads the DE tables we already computed. p-value shown as stars. Kernel: any pandas/matplotlib env.

**Contrast glossary (from the sketch):**
- *Cd-mono vs Cd-co* = Cd alone vs Cd+Bt co-infection (A06) → does the commensal attenuate the IL-8 response. Stored as `Cd_vs_Co`.
- *Cd 0/5/10 mM butyrate* = Cd-vs-Mock at each butyrate dose, and the butyrate contrasts within Cd (A04).

In [3]:
import pandas as pd, numpy as np, re, os
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

A04 = Path(os.environ.get('A04_PROJ', str(Path.home()/'Projects_shared/DECIDE_A04_CdiffOrganoid_Butyrate')))
A06 = Path(os.environ.get('A06_PROJ', str(Path.home()/'Projects_shared/epithelial_coinfection-model_WORKING')))
A04T, A06T = A04/'results/tables', A06/'analysis/DE'
FIG = A04/'results/figures'; FIG.mkdir(parents=True, exist_ok=True)
print('A04 tables:', A04T.exists(), '| A06 DE:', A06T.exists())

def sym_index(df):
    for col in ['gene_name','symbol','gene_symbol','GeneSymbol','Gene','gene']:
        if col in df.columns: df=df.set_index(col); break
    df.index=[re.sub(r'\..*','',str(i)) for i in df.index]
    return df
def load(dirpath,fname):
    f=Path(dirpath)/fname
    if not f.exists(): print('  MISSING',f.name); return None
    return sym_index(pd.read_csv(f,index_col=0))
def getval(d,g):
    if d is None or g not in d.index: return (np.nan,np.nan)
    r=d.loc[g]
    lfc=r.get('log2FoldChange', r.get('log2FC', np.nan))
    padj=r.get('padj', r.get('pvalue', np.nan))
    return float(lfc),float(padj)

A04 tables: True | A06 DE: True


## Gather ALL contrasts for one gene (both studies)

In [5]:
# every A04 table present, plus every A06 pairwise pydeseq2 table
A04_specs = [
  ('DE_Cd_vs_Mock_0mM.csv',  'Cd vs Mock (0mM)'),
  ('DE_Cd_vs_Mock_5mM.csv',  'Cd vs Mock (5mM)'),
  ('DE_Cd_vs_Mock_10mM.csv', 'Cd vs Mock (10mM)'),
  ('DE_Butyrate_5v0_Cd.csv', 'butyrate 5v0 | Cd'),
  ('DE_Butyrate_10v0_Cd.csv','butyrate 10v0 | Cd'),
  ('DE_Butyrate_5v0_Mock.csv','butyrate 5v0 | Mock'),
  ('DE_Butyrate_10v0_Mock.csv','butyrate 10v0 | Mock'),
]
A06_specs = [
  ('pydeseq2_Cd_vs_Mock_annotated.csv','Cd vs Mock'),
  ('pydeseq2_Co_vs_Mock_annotated.csv','Co vs Mock'),
  ('pydeseq2_Cd_vs_Co_annotated.csv',  'Cd-mono vs Cd-co'),   # the sketch bracket
  ('pydeseq2_Bt_vs_Mock_annotated.csv','Bt vs Mock'),
  ('pydeseq2_Bt_vs_Cd_annotated.csv',  'Bt vs Cd'),
  ('pydeseq2_Bt_vs_Co_annotated.csv',  'Bt vs Co'),
  ('pydeseq2_Co_vs_Mock_annotated.csv','Co vs Mock'),
]

def gather(gene):
    rows=[]
    for fn,label in A06_specs:
        d=load(A06T,fn) or load(A06T,fn.replace('_annotated',''))
        lfc,padj=getval(d,gene); rows.append(dict(label=label,study='A06 (co-infection)',lfc=lfc,padj=padj))
    for fn,label in A04_specs:
        d=load(A04T,fn); lfc,padj=getval(d,gene)
        rows.append(dict(label=label,study='A04 (butyrate)',lfc=lfc,padj=padj))
    df=pd.DataFrame(rows).dropna(subset=['lfc']).drop_duplicates('label').reset_index(drop=True)
    return df

cx=gather('CXCL8'); print('CXCL8 contrasts found:',len(cx)); print(cx.round(3).to_string())

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

## Figure 1 — CXCL8 / IL-8 only, all conditions

In [ ]:
def stars(p):
    if pd.isna(p): return 'ns'
    return '***' if p<1e-3 else '**' if p<1e-2 else '*' if p<0.05 else 'ns'

def plot_one(df, gene, fname, color_a06='#3182bd', color_a04='#e6550d'):
    df=df.copy()
    # keep study order: A06 block then A04 block
    df['s']=df.study.str.startswith('A06').map({True:0,False:1})
    df=df.sort_values(['s']).reset_index(drop=True)
    x=range(len(df)); cols=[color_a06 if s==0 else color_a04 for s in df.s]
    fig,ax=plt.subplots(figsize=(max(7,0.85*len(df)+2),4.8))
    ax.bar(x,df.lfc,color=cols,edgecolor='k',lw=.6)
    for xi,(_,r) in zip(x,df.iterrows()):
        off=0.12 if r.lfc>=0 else -0.32
        ax.text(xi,r.lfc+off,stars(r.padj),ha='center',fontsize=9,fontweight='bold')
    ax.axhline(0,color='#333',lw=.8)
    ax.set_xticks(list(x)); ax.set_xticklabels(df.label,rotation=45,ha='right',fontsize=8.5)
    ax.set_ylabel('log2 fold-change',fontsize=10)
    ax.set_title(f'{gene} — log2FC across all conditions',fontsize=13,fontweight='bold')
    ax.grid(axis='y',alpha=.25)
    ax.legend(handles=[Patch(color=color_a06,label='co-infection study'),
                       Patch(color=color_a04,label='butyrate study')],fontsize=8,loc='best')
    ax.text(.99,-.28,'stars: *** p<0.001  ** p<0.01  * p<0.05  ns',transform=ax.transAxes,
            ha='right',fontsize=7,color='#666')
    plt.tight_layout(); fig.savefig(FIG/fname,dpi=150,bbox_inches='tight'); plt.show()
    print('wrote',FIG/fname)

plot_one(cx,'CXCL8 (IL-8)','CXCL8_focus_across_conditions.png')

## Figure 2 — HSPA5 on its own (separate graphic; here because it tracks CXCL8, ρ=0.92)

In [ ]:
hp=gather('HSPA5'); print('HSPA5 contrasts found:',len(hp)); print(hp.round(3).to_string())
plot_one(hp,'HSPA5 (BiP)','HSPA5_across_conditions.png',color_a06='#31a354',color_a04='#756bb1')

## Notes
- Fig 1 is IL-8 only, every available contrast from both studies. "Cd-mono vs Cd-co" is the
  A06 attenuation bar (Cd alone vs Cd+Bt). Expected: IL-8 strongly up with Cd, lower in co-infection
  and with butyrate dose.
- Fig 2 is HSPA5 standalone — shown separately because it is a *correlated readout* (ρ=0.92 with
  CXCL8), not an IL-8 comparison. Its contrast pattern (up with infection, flat/down with butyrate)
  is the visual of the two-arm dissociation.
- Any missing A06 filename prints `MISSING` and is skipped; paste that and I'll fix the path.